# 06 — Distributions
**Goal:** Show distributions of scores, outputs, or metrics — not just mean ± std. A single number hides multi-modality, outliers, and skew.

By the end of this notebook you will:
- Know when to use box plots, violin plots, KDE, strip plots, and ECDF
- Show LLM agent output score distributions across many prompts
- Layer plots for maximum information density
- Apply the right distribution plot for your sample size

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('research.mplstyle')

np.random.seed(11)

# Simulated LLM agent scores across 200 prompts (0-100 scale)
# Model A: unimodal, centered ~70
# Model B: bimodal — either succeeds (~85) or fails (~40)
# Model C: our model — tighter distribution, fewer failures

model_a = np.random.normal(70, 10, 200).clip(0, 100)
model_b = np.concatenate([
    np.random.normal(85, 6, 140),
    np.random.normal(40, 8, 60)
]).clip(0, 100)
model_c = np.random.normal(78, 6, 200).clip(0, 100)

all_models  = [model_a, model_b, model_c]
model_names = ['Model A', 'Model B', 'Ours']
COLORS = ['#BBBBBB', '#BBBBBB', '#0072B2']

---
## Part 1 — Box plot

Box plots show: median, IQR (25th–75th percentile), whiskers (1.5×IQR), and outliers. Good for comparing multiple groups at a glance.

**Weakness:** hides the shape of the distribution (e.g., bimodality).

**Exercise:** Make a box plot comparing all three models.

In [ ]:
# TODO: ax.boxplot(all_models, labels=model_names)
# Style tips:
#   - patch_artist=True  to fill boxes with color
#   - Use boxprops, medianprops, whiskerprops, flierprops to style each element
#   - Hint: ax.boxplot(..., patch_artist=True,
#            boxprops=dict(facecolor='#E0E0E0', linewidth=0.8),
#            medianprops=dict(color='#333333', linewidth=1.5))
# Add y-label 'Score'


---
## Part 2 — Violin plot

Violin plots show the full distribution shape (KDE), not just summary statistics. They reveal bimodality — which box plots hide.

Look at Model B — you should clearly see two modes.

**Exercise:** Make a violin plot for all three models.

In [ ]:
# TODO: ax.violinplot(all_models, positions=[1, 2, 3], showmedians=True)
# The return value is a dict of artists — style each part:
#   parts = ax.violinplot(...)
#   parts['bodies'] is a list of PolyCollection objects — set facecolor, alpha on each
#   parts['cmedians'], parts['cbars'], parts['cmaxes'], parts['cmins'] — set colors
# Set xtick positions and labels to model_names
# Add y-label 'Score'


---
## Part 3 — Strip plot (individual points)

For small samples (N < 50), showing every data point is more honest than a violin. Use `ax.scatter` with jitter (random x-offset) to avoid overplotting.

**Exercise:** Use the first 30 scores from each model and make a strip plot with jitter.

In [ ]:
small_a = model_a[:30]
small_b = model_b[:30]
small_c = model_c[:30]

# TODO: for each model at x position 1, 2, 3:
#   add jitter: x_jittered = x_position + np.random.uniform(-0.15, 0.15, len(scores))
#   ax.scatter(x_jittered, scores, s=20, alpha=0.6, color=...)
# Also add a horizontal line for the mean at each position
#   ax.hlines(mean, x-0.2, x+0.2, linewidth=2, color='#333333')
# Set xtick labels to model_names


---
## Part 4 — KDE plot (kernel density estimate)

KDE is a smoothed version of a histogram — shows the probability density of a continuous variable. Use `scipy.stats.gaussian_kde` or plot overlapping density curves for multiple models.

Great for showing overlapping score distributions in a single axes.

**Exercise:** Overlay KDE curves for all three models on one axes. Fill under each curve with low alpha.

In [ ]:
# TODO: for each model:
#   kde = stats.gaussian_kde(scores)
#   x_range = np.linspace(0, 100, 300)
#   ax.plot(x_range, kde(x_range), color=color, label=name)
#   ax.fill_between(x_range, kde(x_range), alpha=0.12, color=color)

# Make model_b clearly visible — it should show two humps
# Add legend, x-label 'Score', y-label 'Density'


---
## Part 5 — ECDF (Empirical Cumulative Distribution Function)

ECDF is underused but extremely informative. It shows: for any score threshold X, what fraction of prompts did the model score above X?

Formula: sort the data, then `ecdf[i] = i / N`.

**Advantage over KDE:** no smoothing parameter choices; exact representation of your data.

**Exercise:** Plot ECDF curves for all three models.

In [ ]:
def ecdf(data):
    """Return (x, y) for plotting an ECDF."""
    # TODO: sort data
    # TODO: compute y = np.arange(1, len(data)+1) / len(data)
    # return sorted_data, y
    pass

# TODO: plot ECDF for each model
# Add a horizontal reference line at y=0.5 (median)
# Add x-label 'Score', y-label 'Cumulative fraction'
# Legend


---
## Part 6 — Layered violin + strip plot

For medium samples (N = 30–150), a powerful pattern used in top papers: combine violin + individual points. The violin shows shape, the dots show actual data, and a horizontal bar shows the mean.

**Exercise:** Layer all three elements for all three models.

In [ ]:
# Use 60 samples per model for readability
samples = [model_a[:60], model_b[:60], model_c[:60]]

# TODO: draw violins (narrow, alpha=0.4)
# TODO: overlay jittered scatter points (s=15, alpha=0.5) on top
# TODO: overlay mean bar (ax.hlines) for each model
# Set xtick labels, y-label, title
#
# Ordering matters — plot violin first (background), then scatter (foreground)
# Use zorder to control depth: violin zorder=1, scatter zorder=3


---
## When to use what

| Sample size | Best choice |
|-------------|------------|
| N < 30 | Strip plot (show every point) |
| N = 30–150 | Violin + strip overlay |
| N > 150 | Violin or KDE |
| Any N, exact representation | ECDF |
| Quick summary only | Box plot |

**Never use a bar chart to show a distribution** — it throws away almost all the information.

**Next:** `07_multi_panel.ipynb` — composing a full paper figure with subplots, shared axes, and panel labels.